# Discord Persona Chat Interface (Unsloth Optimized)

This notebook runs the fine-tuned Mistral NeMo 12B model. It uses the **Unsloth** backend to double inference speed and minimize VRAM usage, making it perfect for free Colab T4 GPUs.

### Features:
* **Stopping Criteria:** Native C-optimized stop strings prevent the model from talking to itself or roleplaying as the user without massive CPU overhead.
* **Real-Time Streaming:** Uses `TextIteratorStreamer` on a background thread to instantly yield tokens to the screen as they are generated.
* **Message Decimation:** Automatically breaks overly long paragraphs into rapid-fire Discord messages on the fly.
* **Benchmarking:** Use `/benchmark` to automatically run a list of standard questions and save the outputs to your Google Drive.
* **Hot-Swapping:** Use `/switch <gdrive_link> [password]` to dynamically load a different personality without restarting.
* **Dynamic Sliders:** Use `/temp <value>` or `/pen <value>` to change the hallucination/repetition parameters on the fly.


In [ ]:
# ==========================================
# USER CONFIGURATION
# ==========================================

# 1. Adapter Source
GDRIVE_SHARED_LINK = '' # Paste your final_adapter_encrypted.zip link here
DECRYPTION_KEY = ""

# 2. Persona Setup
CLONE_NAME = ""

SYSTEM_PROMPT = f"You are {CLONE_NAME} in a Discord chat."


required_vars = {
    "CLONE_NAME": CLONE_NAME,
    "GDRIVE_SHARED_LINK": GDRIVE_SHARED_LINK
}
missing = [name for name, value in required_vars.items() if not value]
if missing:
    raise ValueError(f"The following variables are required but missing: {', '.join(missing)}")

In [ ]:
import os
import time
import datetime
import json

# Global Logging Wrapper
master_log = ""
def log_print(*args, **kwargs):
    global master_log
    output = " ".join(map(str, args))
    print(output, **kwargs)
    master_log += output + "\n"

# Timing Wrapper
section_times = {}
def start_timer(section_name):
    section_times[section_name] = {'start_time': time.time()}
    start_str = datetime.datetime.fromtimestamp(section_times[section_name]['start_time']).strftime('%H:%M:%S')
    log_print(f"\n[INFO] Started '{section_name}' at {start_str}")

def stop_timer(section_name):
    if section_name in section_times and 'start_time' in section_times[section_name]:
        end_t = time.time()
        duration = end_t - section_times[section_name]['start_time']
        end_str = datetime.datetime.fromtimestamp(end_t).strftime('%H:%M:%S')
        log_print(f"[INFO] Finished '{section_name}' at {end_str} (Took {duration:.2f}s)")


In [ ]:
start_timer("Install Dependencies")
# Install Unsloth and utilities
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q gdown pyzipper modelscope lingua-language-detector
stop_timer("Install Dependencies")

In [ ]:
start_timer("Fetch Adapter")
import gdown
import pyzipper
import shutil

local_zip = '/content/adapter.zip'
local_extract_dir = '/content/local_model'

def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

if GDRIVE_SHARED_LINK:
    log_print("Downloading adapter from Google Drive...")
    file_id = get_gdrive_id(GDRIVE_SHARED_LINK)
    download_url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(download_url, local_zip, quiet=False)

    os.makedirs(local_extract_dir, exist_ok=True)
    log_print("Extracting adapter...")
    try:
        if DECRYPTION_KEY:
            with pyzipper.AESZipFile(local_zip, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                zip_ref.pwd = DECRYPTION_KEY.encode('utf-8')
                zip_ref.extractall(local_extract_dir)
        else:
            with pyzipper.AESZipFile(local_zip, 'r') as zip_ref:
                zip_ref.extractall(local_extract_dir)
        log_print("Adapter successfully extracted.")
    except Exception as e:
        log_print(f"Extraction failed (Check password): {e}")
else:
    log_print("No link provided. Provide a valid link and restart.")

stop_timer("Fetch Adapter")

In [ ]:
start_timer("Load Model (Unsloth Inference Mode)")
import torch
import gc
import os
from unsloth import FastLanguageModel

# Memory cleanup
for var in ['model', 'tokenizer']:
    if var in globals():
        del globals()[var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# --- NEW: Dynamic Path Finder ---
# Hunt down the exact subfolder containing the adapter configuration
adapter_path = local_extract_dir
for root, dirs, files in os.walk(local_extract_dir):
    if 'adapter_config.json' in files:
        adapter_path = root
        break
# --------------------------------

log_print(f"Loading Model via Unsloth from: {adapter_path}")
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1' # Safe fallback

# Unsloth automatically handles loading the base model defined in the adapter's config!
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path, # Uses the dynamically found path
    max_seq_length = 2048, # Giving it plenty of breathing room for inference
    dtype = None, # Auto-detect Float16
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model) # Enables 2x faster inference natively
log_print("Unsloth inference kernels activated.")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

stop_timer("Load Model (Unsloth Inference Mode)")

In [ ]:
import warnings
import transformers
import re
import time
import datetime
import os
import gc # Added back for hot-swapping
from threading import Thread
from transformers import TextIteratorStreamer
from google.colab import drive
from lingua import Language, LanguageDetectorBuilder

# Suppress annoying Hugging Face and PyTorch warnings for a clean chat
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

drive.mount('/content/drive')

BENCHMARK_FILE = '/content/drive/MyDrive/soulclone/custom_benchmarks.txt'

# Build the fast detector specifically for your 3 target languages
languages = [Language.ENGLISH, Language.GERMAN, Language.CZECH]
detector = LanguageDetectorBuilder.from_languages(*languages).build()

def load_benchmarks():
    if os.path.exists(BENCHMARK_FILE):
        with open(BENCHMARK_FILE, 'r') as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
            return lines
    return []

def save_benchmark(prompt):
    os.makedirs(os.path.dirname(BENCHMARK_FILE), exist_ok=True)
    with open(BENCHMARK_FILE, 'a') as f:
        f.write(f"{prompt}\n")

def clear_benchmarks():
    if os.path.exists(BENCHMARK_FILE):
        os.remove(BENCHMARK_FILE)
        return True
    return False

# --- Benchmark Text Loader ---
def import_benchmarks_from_text(filepath):
    if not os.path.exists(filepath):
        print(f"[System: File {filepath} not found.]")
        return False
        
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            
        added_count = 0
        existing = set(load_benchmarks())
        
        for line in lines:
            # Parse the format: "username | prompt"
            if " | " in line:
                prompt = line.split(" | ", 1)[1].strip()
                if prompt and prompt not in existing:
                    save_benchmark(prompt)
                    existing.add(prompt)
                    added_count += 1
        
        print(f"[System: Successfully imported {added_count} new benchmarks from {filepath}]")
        return True
    except Exception as e:
        print(f"[System: Error importing benchmarks: {e}]")
        return False
# -----------------------------

# 1. Post-Generation Response Sanitizer (Used for benchmarks)
def clean_and_format_response(response, clone_name):
    # Strip the symmetrical training prefix so it doesn't double-print
    response = re.sub(r'^\[.*?\]:\s*', '', response.strip())

    # Strip off any leaked username brackets that bypassed native stopping criteria
    if "\n[" in response:
        response = response.split("\n[")[0]
        
    # Clean up bracket hallucinations / context poisoning
    if response.count('(') > 10 or response.count(')') > 10:
        response = response.split('(')[0].split(')')[0] + "... [System: Loop Detected & Cut]"
        
    # Message Decimator: Break huge blocks of text into Discord-like pacing
    if len(response) > 150 and "\n" not in response:
        response = response.replace('. ', '.\n')
        
    return response.strip()

def run_benchmark_suite(model, tokenizer, current_user, temp, pen):
    prompts = load_benchmarks()
    if not prompts:
        print("\n[System: No benchmarks found. Just chat normally, and your prompts will be saved for next time!]")
        return
    
    print(f"\n[System: Running {len(prompts)} stored benchmarks...]")
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = f'/content/drive/MyDrive/soulclone/benchmark_results_{timestamp}.txt'
    
    results = []
    for p in prompts:
        print(f"\n{current_user}: {p}")
        messages = [
            {"role": "system", "content": conversation_history[0]["content"]},
            {"role": "user", "content": f"[{current_user}]: {p}"}
        ]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
        input_length = inputs['input_ids'].shape[1]
        
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150, 
            temperature=temp, 
            repetition_penalty=pen, 
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            tokenizer=tokenizer,
            stop_strings=["\n[", f"\n[{current_user}]"] # Native C-optimized stopping criteria
        )
        response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
        response = clean_and_format_response(response, CLONE_NAME)
        
        print(f"{CLONE_NAME}: {response}")
        results.append(f"{current_user}: {p}\n{CLONE_NAME}: {response}\n")
        
    with open(out_path, 'w') as f:
        f.write("\n".join(results))
    print(f"\n[System: Benchmark results saved to Drive: {out_path}]")

# Start Chat
print("\n" + "*"*65)
print("Clone is ready! Commands:")
print(" 'quit'  : Exit the chat")
print(" 'reset' : Clear memory and change user")
print(" '/benchmark' : Run stored prompts and save outputs to Drive")
print(" '/benchmark --reset' : Wipe stored custom benchmarks")
print(" '/benchmark <filepath>' : Import benchmarks from a text file")
print(" '/temp 0.8' : Change temperature on the fly")
print(" '/pen 1.15' : Change repetition penalty on the fly")
print(" '/lang' : Toggle language injection in system prompt on/off")
print(" '/switch <link> [password]' : Hot-swap to a new persona adapter")
print("*"*65 + "\n")

conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
current_user = input("Enter your username to begin: ").strip()
print(f"\n[Now chatting as {current_user}]\n")

current_temp = 0.75
current_pen = 1.25
use_lang_prompt = False

while True:
    user_input = input(f"{current_user}: ")
    
    # Block empty messages from polluting context
    if not user_input.strip():
        print("[System: You cannot send an empty message. Please type something.]")
        continue
        
    if user_input.lower() in ['quit', 'exit', 'stop']:
        print("Ending chat.")
        break
    elif user_input.lower() == 'reset':
        conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("[Memory cleared.]")
        current_user = input("Enter new username to begin: ").strip()
        print(f"\n[Now chatting as {current_user}]\n")
        continue
    elif user_input.lower() == '/lang':
        use_lang_prompt = not use_lang_prompt
        status = "ON" if use_lang_prompt else "OFF"
        print(f"[System: Language injection is now {status}.]")
        continue
    elif user_input.startswith('/temp '):
        try:
            current_temp = float(user_input.split()[1])
            print(f"[Temperature updated to {current_temp}]")
        except: print("[Invalid temp format. Use: /temp 0.8]")
        continue
    elif user_input.startswith('/pen '):
        try:
            current_pen = float(user_input.split()[1])
            print(f"[Penalty updated to {current_pen}]")
        except: print("[Invalid pen format. Use: /pen 1.15]")
        continue
    elif user_input.startswith('/benchmark'):
        parts = user_input.split()
        if '--reset' in user_input:
            if clear_benchmarks(): print("[System: Custom benchmarks wiped.]")
            else: print("[System: No benchmarks found to wipe.]")
        elif len(parts) > 1 and parts[1] != '--reset':
            filepath = parts[1]
            import_benchmarks_from_text(filepath)
        else:
            run_benchmark_suite(model, tokenizer, current_user, current_temp, current_pen)
        continue
    elif user_input.startswith('/switch'):
        parts = user_input.split()
        if len(parts) < 2:
            print("[System: Usage: /switch <gdrive_link> [password]]")
            continue
        
        new_link = parts[1]
        new_pwd = parts[2] if len(parts) > 2 else ""
        new_zip = '/content/new_adapter.zip'
        new_extract_dir = f'/content/local_model_{int(time.time())}'
        
        print("[System: Hot-swapping adapter via Unsloth. This will take a moment...]")
        try:
            import gdown, pyzipper
            
            def get_gdrive_id(url):
                if '/d/' in url: return url.split('/d/')[1].split('/')[0]
                elif 'id=' in url: return url.split('id=')[1].split('&')[0]
                return url
                
            file_id = get_gdrive_id(new_link)
            gdown.download(f'https://drive.google.com/uc?id={file_id}', new_zip, quiet=True)
            
            os.makedirs(new_extract_dir, exist_ok=True)
            if new_pwd:
                with pyzipper.AESZipFile(new_zip, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                    zip_ref.pwd = new_pwd.encode('utf-8')
                    zip_ref.extractall(new_extract_dir)
            else:
                with pyzipper.AESZipFile(new_zip, 'r') as zip_ref:
                    zip_ref.extractall(new_extract_dir)
            
            adapter_path = new_extract_dir
            for root, dirs, files in os.walk(new_extract_dir):
                if 'adapter_config.json' in files:
                    adapter_path = root
                    break
            
            # Safely clear the old Unsloth model ONLY if it exists
            if 'model' in globals():
                del globals()['model']
            if 'tokenizer' in globals():
                del globals()['tokenizer']
            gc.collect()
            import torch
            torch.cuda.empty_cache()
            
            # Load new model dynamically 
            from unsloth import FastLanguageModel
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name = adapter_path,
                max_seq_length = 2048,
                dtype = None,
                load_in_4bit = True,
            )
            FastLanguageModel.for_inference(model)
            
            print(f"\n[System: Successfully switched to new persona.]")
            conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        except Exception as e:
            print(f"\n[System: Failed to load new adapter: {e}]")
        continue

    # Dynamic Language Detection and System Prompt Injection
    if use_lang_prompt:
        try:
            detected = detector.detect_language_of(user_input)
            target_lang = detected.name.capitalize() if detected else "English"
        except:
            target_lang = "English"
        dynamic_sys_prompt = f"{SYSTEM_PROMPT} Respond in {target_lang}."
    else:
        dynamic_sys_prompt = SYSTEM_PROMPT
        
    conversation_history[0]["content"] = dynamic_sys_prompt

    # Save prompt to benchmarks if it's new
    if user_input not in load_benchmarks():
        save_benchmark(user_input)

    # Format and tokenize
    conversation_history.append({"role": "user", "content": f"[{current_user}]: {user_input}"})
    input_text = tokenizer.apply_chat_template(conversation_history, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    # GPU Background Generation Thread
    generation_kwargs = dict(
        **inputs, 
        max_new_tokens=150, 
        temperature=current_temp, 
        repetition_penalty=current_pen, 
        do_sample=True,
        top_p=1.0,
        min_p=0.05,
        pad_token_id=tokenizer.eos_token_id,
        tokenizer=tokenizer,
        stop_strings=["\n[", f"\n[{current_user}]"], # Native stopping criteria
        streamer=streamer
    )
    
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    # Live Printing loop (Fake Streaming)
    print(f"{CLONE_NAME}: ", end="")
    generated_text = ""
    prefix_stripped = False
    
    for new_text in streamer:
        generated_text += new_text
        # Intercept and hide the symmetrical training prefix live
        if not prefix_stripped:
            if "]:" in generated_text:
                clean_text = re.sub(r'^\[.*?\]:\s*', '', generated_text)
                print(clean_text, end="", flush=True)
                prefix_stripped = True
            elif len(generated_text) > 25 and not generated_text.startswith("["):
                print(generated_text, end="", flush=True)
                prefix_stripped = True
        else:
            print(new_text, end="", flush=True)
            time.sleep(0.04) # Mimics typing delay
            
    print()
    
    # Post-generation cleanup before saving to context memory
    if "\n[" in generated_text:
        generated_text = generated_text.split("\n[")[0]
        
    conversation_history.append({"role": "assistant", "content": generated_text.strip()})
    
    # Dynamic Token-Aware Memory Manager (Cap at ~500 tokens for T4 VRAM safety)
    while True:
        # Temporarily apply template to check exact token length
        test_str = tokenizer.apply_chat_template(conversation_history, tokenize=False)
        token_count = len(tokenizer.encode(test_str))
        
        if token_count > 500:
            # Memory is too full. Pop the oldest User/Assistant pair (indices 1 and 2)
            # We leave index 0 intact because it is the System Prompt
            if len(conversation_history) > 3:
                conversation_history.pop(1)
                conversation_history.pop(1)
            else:
                break # Edge case: single turn is >500 tokens
        else:
            break # Safe token limit reached